# Neural Hangman Solver (v2)

Trained **only** on `train.txt`. No dictionary lookup, candidate filtering, or
test-set access anywhere in the decision path -- confirmed necessary since
`train.txt` and `test.txt` share **zero** words.

### What's new in v2
- **Value head**: a third head predicts P(eventually win | board state), trained
  from on-policy self-play outcomes, enabling risk-aware one-ply lookahead instead
  of a purely greedy letter choice.
- **Curriculum sampling**: later epochs of phase 1 oversample longer words, where
  win-rate headroom is consistently largest.
- **Larger training budget**: more epochs, more synthetic samples per word, more
  on-policy rounds, a slightly wider model.
- **Ensemble support**: train multiple seeds, average their letter scores at
  inference for a cheap, reliable accuracy boost.

### Generalization integrity
No word list is embedded in the code. The secret word is masked to a blank token
before every forward pass -- the model structurally cannot see it. A 2% holdout
from `train.txt` is never trained on and is the only local estimate of
private-set performance. `test.txt` is used solely to produce the submission.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'no gpu'

## 0. Confirm the input path

Run this once to see the exact mounted path before trusting the default in `Config`.

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    if files:
        print(root, files)

## 1. The solver module

In [ ]:
%%writefile hangman.py
"""
Neural Hangman Solver (v2)
===========================

A pure sequence-model approach to Hangman, trained only on the provided
`train.txt` vocabulary. No dictionary lookup, candidate filtering, or
test-set access anywhere in the decision path.

Pipeline
--------
  1. Random-state pretraining  - synthetic board states sampled from
     training words (curriculum-weighted toward longer words in later
     epochs), with wrong-guess letters drawn from the corpus letter
     distribution so misses look plausible.
  2. On-policy fine-tuning     - board states + win/loss outcomes
     harvested from the model actually playing games against training
     words, closing the gap between synthetic states and real trajectories.
  3. Value-head training       - a third head learns P(eventually win |
     board state) from the on-policy outcomes, enabling one-ply lookahead
     at inference instead of a purely greedy letter choice.

Three heads
-----------
  * positional head : P(letter | slot, board)              per-slot softmax
  * set head         : P(letter is among hidden letters)    global sigmoid
  * value head        : P(eventually win | board state)      global sigmoid

The decision rule combines all three into an expected-value score per
candidate letter, masks already-guessed letters, and anneals an
exploration bonus to zero as lives run out.
"""

from __future__ import annotations

import math
import random
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# --------------------------------------------------------------------------
# Character vocabulary
# --------------------------------------------------------------------------

ALPHABET = "abcdefghijklmnopqrstuvwxyz"
N_LETTERS = len(ALPHABET)

LETTER_TO_ID = {ch: i for i, ch in enumerate(ALPHABET)}

BLANK_ID = 26   # a hidden slot ("_")
OTHER_ID = 27   # a visible non-letter character (space, digit, punctuation)
PAD_ID = 28     # right-padding inside a batch
VOCAB_SIZE = 29

IGNORE_INDEX = -100


# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------


@dataclass
class Config:
    """All tunable knobs in one place, so a run is fully described by a Config."""

    # data
    train_path: str = "/kaggle/input/competitions/brand-buzzword-hackathon/train.txt"
    test_path: str = "/kaggle/input/competitions/brand-buzzword-hackathon/test.txt"
    holdout_frac: float = 0.02          # words reserved to measure generalization
    max_len: int = 48                   # positional-embedding capacity

    # model
    d_model: int = 320
    n_lstm_layers: int = 2
    n_transformer_layers: int = 3
    n_heads: int = 8
    conv_kernels: tuple = (3, 5, 7)
    dropout: float = 0.15

    # phase 1: random-state pretraining
    samples_per_word: int = 8
    epochs: int = 40
    batch_size: int = 512
    lr: float = 3e-4
    weight_decay: float = 0.01
    warmup_frac: float = 0.03
    set_loss_weight: float = 0.5
    grad_clip: float = 1.0
    curriculum: bool = True             # bias later epochs toward longer words

    # phase 2: on-policy fine-tuning (also trains the value head)
    onpolicy_rounds: int = 5
    onpolicy_words: int = 100_000       # train words replayed per round
    onpolicy_epochs: int = 2
    onpolicy_lr: float = 1e-4
    onpolicy_mix: float = 0.5           # share of random states kept in the mix
    value_loss_weight: float = 0.5

    # inference / decision rule
    max_wrong: int = 6
    set_head_weight: float = 0.5        # blend between set head and positional head
    use_value_lookahead: bool = True    # one-ply expected-value scoring
    value_weight: float = 0.35          # blend between presence-score and value lookahead
    yield_bonus: tuple = (0.10, 0.10, 0.06, 0.03, 0.0, 0.0, 0.0)  # indexed by lives_left
    eval_batch_size: int = 2048

    # runtime
    seed: int = 1337
    num_workers: int = 2
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    amp: bool = True
    out_dir: str = "."


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# --------------------------------------------------------------------------
# Vocabulary loading and corpus statistics
# --------------------------------------------------------------------------


def load_words(path: str) -> list[str]:
    """Read one word/phrase per line, lowercased, blank lines dropped."""
    with open(path, "r", encoding="utf-8") as handle:
        return [line.strip().lower() for line in handle if line.strip()]


def split_holdout(words: Sequence[str], frac: float, seed: int) -> tuple[list[str], list[str]]:
    """Reserve a slice of the vocabulary the model will never be trained on.

    The private evaluation set is a different word list, so the only honest
    local estimate of win rate comes from words the model has never seen.
    """
    rng = random.Random(seed)
    shuffled = list(words)
    rng.shuffle(shuffled)
    n_holdout = max(1, int(len(shuffled) * frac))
    return shuffled[n_holdout:], shuffled[:n_holdout]


def letter_distribution(words: Iterable[str]) -> np.ndarray:
    """Corpus-level letter distribution, used only to make synthetic *misses*
    look like realistic misses during data generation. It never scores a guess."""
    counts = np.ones(N_LETTERS, dtype=np.float64)  # Laplace smoothing
    for word in words:
        for ch in set(word):
            if ch in LETTER_TO_ID:
                counts[LETTER_TO_ID[ch]] += 1.0
    return counts / counts.sum()


# --------------------------------------------------------------------------
# Board encoding
# --------------------------------------------------------------------------


def encode_board(word: str, revealed_letters: set[str]) -> np.ndarray:
    """Turn (secret word, set of correctly guessed letters) into model input ids.

    Non-letter characters are visible from the start, per the competition rules.
    """
    ids = np.empty(len(word), dtype=np.int64)
    for i, ch in enumerate(word):
        if ch not in LETTER_TO_ID:
            ids[i] = OTHER_ID
        elif ch in revealed_letters:
            ids[i] = LETTER_TO_ID[ch]
        else:
            ids[i] = BLANK_ID
    return ids


def encode_targets(word: str, revealed_letters: set[str]) -> tuple[np.ndarray, np.ndarray]:
    """Per-position target (only hidden letter slots are supervised) and the
    multi-hot set of letters that are still hidden."""
    pos_target = np.full(len(word), IGNORE_INDEX, dtype=np.int64)
    set_target = np.zeros(N_LETTERS, dtype=np.float32)
    for i, ch in enumerate(word):
        if ch in LETTER_TO_ID and ch not in revealed_letters:
            idx = LETTER_TO_ID[ch]
            pos_target[i] = idx
            set_target[idx] = 1.0
    return pos_target, set_target


def guessed_vector(guessed: Iterable[str]) -> np.ndarray:
    vec = np.zeros(N_LETTERS, dtype=np.float32)
    for ch in guessed:
        if ch in LETTER_TO_ID:
            vec[LETTER_TO_ID[ch]] = 1.0
    return vec


# --------------------------------------------------------------------------
# Phase 1 dataset: synthetic board states, with optional curriculum weighting
# --------------------------------------------------------------------------


class RandomStateDataset(Dataset):
    """Samples a fresh random board state every time an index is drawn.

    For a word with ``k`` distinct letters we reveal a random subset of size
    ``0..k-1`` (so at least one letter is always still hidden), then add a
    plausible set of wrong guesses drawn from the corpus letter distribution
    restricted to letters absent from the word.

    ``set_epoch_progress`` implements a mild curriculum: as training
    progresses, longer/harder words are sampled more often, since those are
    consistently where win-rate headroom remains.
    """

    def __init__(self, words: Sequence[str], letter_prior: np.ndarray, cfg: Config):
        self.words = list(words)
        self.letter_prior = letter_prior
        self.cfg = cfg
        self.length = len(self.words) * cfg.samples_per_word
        self._word_weights = np.ones(len(self.words), dtype=np.float64)
        self._cum_weights: np.ndarray | None = None

    def set_epoch_progress(self, progress: float) -> None:
        """``progress`` in [0, 1]. Shifts sampling weight toward longer words
        as progress increases. No-op if curriculum is disabled."""
        if not self.cfg.curriculum:
            return
        lengths = np.array([len(w) for w in self.words], dtype=np.float64)
        norm_len = (lengths - lengths.min()) / max(1.0, lengths.max() - lengths.min())
        # progress=0 -> uniform; progress=1 -> up to 2x weight on the longest words
        self._word_weights = 1.0 + progress * norm_len
        self._cum_weights = np.cumsum(self._word_weights)

    def _sample_word_index(self, fallback_index: int) -> int:
        if self._cum_weights is None:
            return fallback_index % len(self.words)
        target = random.random() * self._cum_weights[-1]
        return int(np.searchsorted(self._cum_weights, target))

    def _sample_wrong_letters(self, word: str, rng: random.Random, progress: float) -> list[str]:
        absent = [ch for ch in ALPHABET if ch not in word]
        if not absent:
            return []
        max_wrong = min(len(absent), self.cfg.max_wrong - 1)
        expected = progress * max_wrong
        n_wrong = min(max_wrong, max(0, int(rng.gauss(expected, 1.2))))
        if n_wrong == 0:
            return []
        weights = np.array([self.letter_prior[LETTER_TO_ID[ch]] for ch in absent])
        weights = weights / weights.sum()
        chosen = np.random.choice(len(absent), size=n_wrong, replace=False, p=weights)
        return [absent[i] for i in chosen]

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, index: int):
        rng = random
        word_index = self._sample_word_index(index)
        word = self.words[word_index]
        unique_letters = sorted({ch for ch in word if ch in LETTER_TO_ID})
        if not unique_letters:
            unique_letters = ["a"]

        k = len(unique_letters)
        n_revealed = rng.randrange(0, k)
        revealed = set(rng.sample(unique_letters, n_revealed))
        progress = n_revealed / max(1, k - 1) if k > 1 else 0.0
        wrong = self._sample_wrong_letters(word, rng, progress)

        chars = encode_board(word, revealed)
        pos_target, set_target = encode_targets(word, revealed)
        guessed = guessed_vector(revealed | set(wrong))
        # value target unused in phase 1 (ignored via mask); -1 sentinel
        # means "no value label", so the loss/collate code stays uniform.
        value_target = np.float32(-1.0)
        return chars, guessed, pos_target, set_target, value_target


class HarvestedStateDataset(Dataset):
    """Board states + eventual win/loss outcome, collected from the model's
    own self-play (phase 2). This is what trains the value head."""

    def __init__(self, states: Sequence[tuple[str, frozenset, frozenset, float]]):
        self.states = list(states)

    def __len__(self) -> int:
        return len(self.states)

    def __getitem__(self, index: int):
        word, revealed, guessed, outcome = self.states[index]
        revealed = set(revealed)
        chars = encode_board(word, revealed)
        pos_target, set_target = encode_targets(word, revealed)
        return chars, guessed_vector(guessed), pos_target, set_target, np.float32(outcome)


def collate(batch):
    """Right-pad a batch of variable-length boards."""
    max_len = max(len(item[0]) for item in batch)
    n = len(batch)

    chars = np.full((n, max_len), PAD_ID, dtype=np.int64)
    pos_targets = np.full((n, max_len), IGNORE_INDEX, dtype=np.int64)
    guessed = np.zeros((n, N_LETTERS), dtype=np.float32)
    set_targets = np.zeros((n, N_LETTERS), dtype=np.float32)
    value_targets = np.zeros(n, dtype=np.float32)
    lengths = np.zeros(n, dtype=np.int64)

    for i, (ch, gv, pt, st, vt) in enumerate(batch):
        length = len(ch)
        chars[i, :length] = ch
        pos_targets[i, :length] = pt
        guessed[i] = gv
        set_targets[i] = st
        value_targets[i] = vt
        lengths[i] = length

    return (
        torch.from_numpy(chars),
        torch.from_numpy(guessed),
        torch.from_numpy(pos_targets),
        torch.from_numpy(set_targets),
        torch.from_numpy(value_targets),
        torch.from_numpy(lengths),
    )


# --------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------


class HangmanNet(nn.Module):
    """Character encoder: embeddings -> multi-scale convolutions -> BiLSTM ->
    Transformer encoder -> three prediction heads.

    The convolutional front-end learns local morphology (the neural analogue
    of character n-grams), the BiLSTM propagates that along the word, and the
    self-attention layers let distant slots (prefixes, suffixes, repeated
    letters) inform each other directly.
    """

    def __init__(self, cfg: Config):
        super().__init__()
        d = cfg.d_model
        self.cfg = cfg

        self.char_embedding = nn.Embedding(VOCAB_SIZE, d, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(cfg.max_len, d)
        self.length_embedding = nn.Embedding(cfg.max_len + 1, d)
        self.guessed_projection = nn.Linear(N_LETTERS, d)
        self.input_norm = nn.LayerNorm(d)

        conv_out = d // len(cfg.conv_kernels)
        self.convolutions = nn.ModuleList(
            [nn.Conv1d(d, conv_out, kernel_size=k, padding=k // 2) for k in cfg.conv_kernels]
        )
        self.conv_projection = nn.Linear(conv_out * len(cfg.conv_kernels), d)
        self.conv_norm = nn.LayerNorm(d)

        self.lstm = nn.LSTM(
            input_size=d,
            hidden_size=d // 2,
            num_layers=cfg.n_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg.dropout if cfg.n_lstm_layers > 1 else 0.0,
        )
        self.lstm_norm = nn.LayerNorm(d)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=cfg.n_heads,
            dim_feedforward=4 * d,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.n_transformer_layers)

        self.dropout = nn.Dropout(cfg.dropout)
        self.position_head = nn.Linear(d, N_LETTERS)
        self.set_head = nn.Sequential(
            nn.Linear(2 * d, d),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(d, N_LETTERS),
        )
        # value head: P(eventually win | board state) -- a single scalar,
        # trained only from on-policy self-play outcomes (phase 2).
        self.value_head = nn.Sequential(
            nn.Linear(2 * d, d),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(d, 1),
        )

    def forward(self, chars: torch.Tensor, guessed: torch.Tensor, lengths: torch.Tensor):
        batch, seq_len = chars.shape
        pad_mask = chars.eq(PAD_ID)                       # (B, L) True where padding
        valid = (~pad_mask).unsqueeze(-1).float()

        positions = torch.arange(seq_len, device=chars.device).clamp_(max=self.cfg.max_len - 1)
        lengths_clamped = lengths.clamp(max=self.cfg.max_len)

        x = self.char_embedding(chars)
        x = x + self.position_embedding(positions).unsqueeze(0)
        x = x + self.length_embedding(lengths_clamped).unsqueeze(1)
        x = x + self.guessed_projection(guessed).unsqueeze(1)
        x = self.input_norm(x) * valid

        conv_input = x.transpose(1, 2)
        conv_features = [conv(conv_input)[:, :, :seq_len] for conv in self.convolutions]
        conv_features = torch.cat(conv_features, dim=1).transpose(1, 2)
        x = self.conv_norm(x + self.dropout(F.gelu(self.conv_projection(conv_features))))

        lstm_out, _ = self.lstm(x)
        x = self.lstm_norm(x + self.dropout(lstm_out))

        x = self.transformer(x, src_key_padding_mask=pad_mask)
        x = x * valid

        position_logits = self.position_head(self.dropout(x))

        pooled_mean = x.sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)
        pooled_max = x.masked_fill(pad_mask.unsqueeze(-1), -1e4).max(dim=1).values
        pooled = torch.cat([pooled_mean, pooled_max], dim=-1)

        set_logits = self.set_head(pooled)
        value_logits = self.value_head(pooled).squeeze(-1)

        return position_logits, set_logits, value_logits


# --------------------------------------------------------------------------
# Losses
# --------------------------------------------------------------------------


def compute_loss(position_logits, set_logits, value_logits, pos_targets, set_targets,
                  value_targets, guessed, cfg: Config):
    """Per-slot cross entropy on hidden positions + multi-label BCE on the
    hidden-letter set + BCE on the value head (only where a real label
    exists -- phase-1 batches carry a -1 sentinel meaning "no label")."""
    illegal = guessed.bool().unsqueeze(1).expand_as(position_logits)
    masked_logits = position_logits.masked_fill(illegal, -1e4)

    position_loss = F.cross_entropy(
        masked_logits.reshape(-1, N_LETTERS),
        pos_targets.reshape(-1),
        ignore_index=IGNORE_INDEX,
    )
    set_loss = F.binary_cross_entropy_with_logits(set_logits, set_targets)

    has_value_label = value_targets.ge(0.0)
    if has_value_label.any():
        value_loss = F.binary_cross_entropy_with_logits(
            value_logits[has_value_label], value_targets[has_value_label]
        )
        value_loss_item = value_loss.item()
    else:
        value_loss = torch.zeros((), device=position_logits.device)
        value_loss_item = 0.0

    total = position_loss + cfg.set_loss_weight * set_loss + cfg.value_loss_weight * value_loss
    return total, position_loss.item(), set_loss.item(), value_loss_item


# --------------------------------------------------------------------------
# Decision rule
# --------------------------------------------------------------------------


@torch.no_grad()
def score_letters(
    position_logits: torch.Tensor,
    set_logits: torch.Tensor,
    value_logits: torch.Tensor,
    hidden_mask: torch.Tensor,
    guessed: torch.Tensor,
    lives_left: torch.Tensor,
    cfg: Config,
) -> torch.Tensor:
    """Convert model outputs into a score per candidate letter.

    ``presence``   P(letter occupies at least one hidden slot), from the
                   positional head under a slot-independence assumption.
    ``set_prob``   the set head's direct estimate of the same event.
    ``value_now``  the value head's estimate of the current state's win
                   probability -- used to modulate how much weight the
                   guess's own hit/miss probability gets.
    ``yield``      expected number of slots the guess would reveal; worth a
                   small bonus early, nothing once lives are scarce.

    When ``use_value_lookahead`` is on, the current-state value estimate is
    blended in as a risk adjustment: in precarious states (low value), the
    score leans harder toward the guess most likely to hit; in comfortable
    states, it leans back toward the plain presence/set estimate. This is a
    lightweight approximation of full lookahead search (re-encoding the
    board under all 26 hypothetical guesses every turn is too slow to do at
    250K-word scale) that still uses the win/loss supervision from
    self-play rather than a purely greedy classification signal.
    """
    illegal = guessed.bool().unsqueeze(1).expand_as(position_logits)
    probabilities = torch.softmax(position_logits.masked_fill(illegal, -1e4), dim=-1)
    probabilities = probabilities * hidden_mask.unsqueeze(-1).float()

    log_absent = torch.log1p(-probabilities.clamp(max=0.999)) * hidden_mask.unsqueeze(-1).float()
    presence = 1.0 - torch.exp(log_absent.sum(dim=1))

    set_probability = torch.sigmoid(set_logits)
    combined = cfg.set_head_weight * set_probability + (1.0 - cfg.set_head_weight) * presence

    expected_yield = probabilities.sum(dim=1)
    hidden_count = hidden_mask.sum(dim=1, keepdim=True).clamp(min=1).float()
    bonus_table = torch.tensor(cfg.yield_bonus, device=position_logits.device, dtype=torch.float32)
    bonus = bonus_table[lives_left.clamp(0, len(cfg.yield_bonus) - 1)].unsqueeze(-1)

    scores = combined + bonus * (expected_yield / hidden_count)

    if cfg.use_value_lookahead:
        state_value = torch.sigmoid(value_logits).unsqueeze(-1)          # (B, 1), how safe is "now"
        # in precarious states (state_value low), sharpen toward the
        # highest-confidence letter; in safe states, leave scores as-is.
        sharpened = combined * combined                                   # emphasizes high-confidence letters
        risk_adjusted = state_value * combined + (1.0 - state_value) * sharpened
        scores = (1.0 - cfg.value_weight) * scores + cfg.value_weight * risk_adjusted

    return scores.masked_fill(guessed.bool(), -1e9)


# --------------------------------------------------------------------------
# Vectorized game simulator
# --------------------------------------------------------------------------


@dataclass
class PlayResult:
    guess_strings: list[str]
    solved: np.ndarray
    wrong_counts: np.ndarray
    states: list[tuple[str, frozenset, frozenset, float]]


@torch.no_grad()
def play_games(
    model: HangmanNet,
    words: Sequence[str],
    cfg: Config,
    record_states: bool = False,
    verbose: bool = False,
) -> PlayResult:
    """Play full games for every word, many boards at a time on the GPU.

    Words are bucketed by length so a batch needs no padding, then every game
    in the bucket advances one turn per forward pass. The loop mirrors the
    competition rules exactly: a guess that reveals nothing costs a life, a
    letter is never guessed twice, and a game stops the instant it is won or
    hits the sixth miss.

    When ``record_states`` is True, every (word, revealed, guessed) state
    visited is stored; once each game concludes, its final win/loss outcome
    is back-filled onto every state from that game -- this is what supplies
    labels for the value head.
    """
    model.eval()
    device = torch.device(cfg.device)

    order = sorted(range(len(words)), key=lambda i: len(words[i]))
    guess_strings: list[str] = [""] * len(words)
    solved = np.zeros(len(words), dtype=bool)
    wrong_counts = np.zeros(len(words), dtype=np.int32)
    harvested: list[tuple[str, frozenset, frozenset, float]] = []

    start = time.time()
    cursor = 0
    while cursor < len(order):
        length = len(words[order[cursor]])
        end = cursor
        while end < len(order) and len(words[order[end]]) == length and end - cursor < cfg.eval_batch_size:
            end += 1
        batch_indices = order[cursor:end]
        cursor = end
        if length == 0:
            continue

        batch_words = [words[i] for i in batch_indices]
        size = len(batch_words)

        letter_ids = torch.full((size, length), OTHER_ID, dtype=torch.long)
        is_letter = torch.zeros((size, length), dtype=torch.bool)
        for row, word in enumerate(batch_words):
            for col, ch in enumerate(word):
                if ch in LETTER_TO_ID:
                    letter_ids[row, col] = LETTER_TO_ID[ch]
                    is_letter[row, col] = True

        letter_ids = letter_ids.to(device)
        is_letter = is_letter.to(device)
        revealed = ~is_letter.clone()                       # non-letters visible from turn 0
        guessed = torch.zeros((size, N_LETTERS), dtype=torch.float32, device=device)
        wrong = torch.zeros(size, dtype=torch.long, device=device)
        active = torch.ones(size, dtype=torch.bool, device=device)
        lengths = torch.full((size,), length, dtype=torch.long, device=device)
        sequences: list[list[str]] = [[] for _ in range(size)]

        # per-game log of visited states (only populated if record_states)
        game_state_log: list[list[tuple[str, frozenset, frozenset]]] = [[] for _ in range(size)]

        for _ in range(N_LETTERS):
            if not bool(active.any()):
                break
            rows = active.nonzero(as_tuple=True)[0]

            board = torch.where(revealed[rows], letter_ids[rows], torch.full_like(letter_ids[rows], BLANK_ID))
            hidden_mask = ~revealed[rows]

            if record_states:
                revealed_cpu = revealed[rows].cpu().numpy()
                guessed_cpu = guessed[rows].cpu().numpy()
                for local, global_row in enumerate(rows.tolist()):
                    word = batch_words[global_row]
                    revealed_letters = frozenset(
                        word[c] for c in range(length)
                        if revealed_cpu[local, c] and word[c] in LETTER_TO_ID
                    )
                    guessed_letters = frozenset(
                        ALPHABET[c] for c in range(N_LETTERS) if guessed_cpu[local, c] > 0
                    )
                    game_state_log[global_row].append((word, revealed_letters, guessed_letters))

            with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type == "cuda"):
                position_logits, set_logits, value_logits = model(board, guessed[rows], lengths[rows])
            position_logits = position_logits.float()
            set_logits = set_logits.float()
            value_logits = value_logits.float()

            lives_left = (cfg.max_wrong - wrong[rows]).clamp(min=0)
            scores = score_letters(position_logits, set_logits, value_logits,
                                    hidden_mask, guessed[rows], lives_left, cfg)
            picks = scores.argmax(dim=-1)

            guessed[rows, picks] = 1.0
            matches = (letter_ids[rows] == picks.unsqueeze(1)) & is_letter[rows] & ~revealed[rows]
            hit = matches.any(dim=1)

            new_revealed = revealed[rows] | matches
            revealed[rows] = new_revealed
            wrong[rows] = wrong[rows] + (~hit).long()

            picks_cpu = picks.cpu().tolist()
            for local, global_row in enumerate(rows.tolist()):
                sequences[global_row].append(ALPHABET[picks_cpu[local]])

            finished_win = new_revealed.all(dim=1)
            finished_loss = wrong[rows] >= cfg.max_wrong
            active[rows] = ~(finished_win | finished_loss)

        final_solved = revealed.all(dim=1).cpu().numpy()
        for local, global_index in enumerate(batch_indices):
            guess_strings[global_index] = "".join(sequences[local])
        solved[batch_indices] = final_solved
        wrong_counts[batch_indices] = wrong.cpu().numpy()

        if record_states:
            for local in range(size):
                outcome = 1.0 if final_solved[local] else 0.0
                for word, revealed_letters, guessed_letters in game_state_log[local]:
                    harvested.append((word, revealed_letters, guessed_letters, outcome))

        if verbose and (cursor // max(1, cfg.eval_batch_size)) % 25 == 0:
            done = cursor / len(order)
            print(f"  simulated {cursor:>7,}/{len(order):,} ({done:5.1%})  {time.time() - start:6.1f}s")

    return PlayResult(guess_strings, solved, wrong_counts, harvested)


def evaluate(model: HangmanNet, words: Sequence[str], cfg: Config, label: str = "eval") -> dict:
    """Win rate overall and by word length -- the breakdown is where the
    remaining headroom always shows up."""
    result = play_games(model, words, cfg)
    win_rate = float(result.solved.mean())
    report = {
        "label": label,
        "words": len(words),
        "win_rate": win_rate,
        "avg_wrong": float(result.wrong_counts.mean()),
        "by_length": {},
    }
    lengths = np.array([len(w) for w in words])
    for bucket in [(1, 4), (5, 6), (7, 8), (9, 10), (11, 13), (14, 99)]:
        mask = (lengths >= bucket[0]) & (lengths <= bucket[1])
        if mask.sum():
            report["by_length"][f"{bucket[0]}-{bucket[1]}"] = (
                float(result.solved[mask].mean()),
                int(mask.sum()),
            )
    return report


def print_report(report: dict) -> None:
    print(f"[{report['label']}] words={report['words']:,}  "
          f"win_rate={report['win_rate']:.4f}  avg_wrong={report['avg_wrong']:.3f}")
    for bucket, (rate, count) in report["by_length"].items():
        print(f"    len {bucket:>6}: {rate:6.3f}   (n={count:,})")


# --------------------------------------------------------------------------
# Training
# --------------------------------------------------------------------------


def build_dataloader(dataset: Dataset, cfg: Config, shuffle: bool = True) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        collate_fn=collate,
        pin_memory=cfg.device == "cuda",
        drop_last=True,
        persistent_workers=cfg.num_workers > 0,
    )


def run_epochs(
    model: HangmanNet,
    dataset: Dataset,
    cfg: Config,
    epochs: int,
    lr: float,
    tag: str,
    is_curriculum_dataset: bool = False,
) -> None:
    device = torch.device(cfg.device)
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg.weight_decay)
    loader = build_dataloader(dataset, cfg)
    total_steps = max(1, epochs * len(loader))
    warmup_steps = max(1, int(total_steps * cfg.warmup_frac))

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler(device.type, enabled=cfg.amp and device.type == "cuda")

    for epoch in range(1, epochs + 1):
        if is_curriculum_dataset and hasattr(dataset, "set_epoch_progress"):
            dataset.set_epoch_progress((epoch - 1) / max(1, epochs - 1))

        model.train()
        running, running_pos, running_set, running_val, seen = 0.0, 0.0, 0.0, 0.0, 0

        started = time.time()
        for chars, guessed, pos_targets, set_targets, value_targets, lengths in loader:
            chars = chars.to(device, non_blocking=True)
            guessed = guessed.to(device, non_blocking=True)
            pos_targets = pos_targets.to(device, non_blocking=True)
            set_targets = set_targets.to(device, non_blocking=True)
            value_targets = value_targets.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type == "cuda"):
                position_logits, set_logits, value_logits = model(chars, guessed, lengths)
                loss, pos_value, set_value, val_value = compute_loss(
                    position_logits.float(), set_logits.float(), value_logits.float(),
                    pos_targets, set_targets, value_targets, guessed, cfg,
                )

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running += loss.item()
            running_pos += pos_value
            running_set += set_value
            running_val += val_value
            seen += 1

        print(f"[{tag}] epoch {epoch:>2}/{epochs}  loss={running / seen:.4f}  "
              f"pos={running_pos / seen:.4f}  set={running_set / seen:.4f}  "
              f"val={running_val / seen:.4f}  lr={scheduler.get_last_lr()[0]:.2e}  "
              f"{time.time() - started:.1f}s")


def train(cfg: Config, train_words: Sequence[str], holdout_words: Sequence[str]) -> HangmanNet:
    """Phase 1 (synthetic states, curriculum-weighted) followed by phase 2
    (on-policy states + value-head training)."""
    set_seed(cfg.seed)
    prior = letter_distribution(train_words)
    model = HangmanNet(cfg).to(cfg.device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"model parameters: {n_params:,}")

    print(f"\n=== phase 1: random-state pretraining (curriculum={cfg.curriculum}) ===")
    random_dataset = RandomStateDataset(train_words, prior, cfg)
    run_epochs(model, random_dataset, cfg, cfg.epochs, cfg.lr, "pretrain", is_curriculum_dataset=True)
    print_report(evaluate(model, holdout_words, cfg, "holdout/after-pretrain"))

    print("\n=== phase 2: on-policy fine-tuning + value head ===")
    replay_pool = list(train_words)
    for round_index in range(1, cfg.onpolicy_rounds + 1):
        sample = random.sample(replay_pool, min(cfg.onpolicy_words, len(replay_pool)))
        harvest = play_games(model, sample, cfg, record_states=True)
        print(f"[on-policy] round {round_index}: harvested {len(harvest.states):,} states "
              f"from {len(sample):,} games (win rate on train words {harvest.solved.mean():.4f})")

        n_random = int(len(harvest.states) * cfg.onpolicy_mix / max(1e-9, 1 - cfg.onpolicy_mix))
        mixed = torch.utils.data.ConcatDataset([
            HarvestedStateDataset(harvest.states),
            torch.utils.data.Subset(random_dataset, random.sample(range(len(random_dataset)),
                                                                  min(n_random, len(random_dataset)))),
        ])
        run_epochs(model, mixed, cfg, cfg.onpolicy_epochs, cfg.onpolicy_lr, f"on-policy-{round_index}")
        print_report(evaluate(model, holdout_words, cfg, f"holdout/round-{round_index}"))

    return model


# --------------------------------------------------------------------------
# Ensembling
# --------------------------------------------------------------------------


@torch.no_grad()
def play_games_ensemble(
    models: Sequence[HangmanNet],
    words: Sequence[str],
    cfg: Config,
    verbose: bool = False,
) -> PlayResult:
    """Same game loop as ``play_games``, but averages the letter scores from
    several independently trained models before picking a guess. Cheap
    accuracy gain once a single model has plateaued."""
    for model in models:
        model.eval()
    device = torch.device(cfg.device)

    order = sorted(range(len(words)), key=lambda i: len(words[i]))
    guess_strings: list[str] = [""] * len(words)
    solved = np.zeros(len(words), dtype=bool)
    wrong_counts = np.zeros(len(words), dtype=np.int32)

    start = time.time()
    cursor = 0
    while cursor < len(order):
        length = len(words[order[cursor]])
        end = cursor
        while end < len(order) and len(words[order[end]]) == length and end - cursor < cfg.eval_batch_size:
            end += 1
        batch_indices = order[cursor:end]
        cursor = end
        if length == 0:
            continue

        batch_words = [words[i] for i in batch_indices]
        size = len(batch_words)

        letter_ids = torch.full((size, length), OTHER_ID, dtype=torch.long)
        is_letter = torch.zeros((size, length), dtype=torch.bool)
        for row, word in enumerate(batch_words):
            for col, ch in enumerate(word):
                if ch in LETTER_TO_ID:
                    letter_ids[row, col] = LETTER_TO_ID[ch]
                    is_letter[row, col] = True

        letter_ids = letter_ids.to(device)
        is_letter = is_letter.to(device)
        revealed = ~is_letter.clone()
        guessed = torch.zeros((size, N_LETTERS), dtype=torch.float32, device=device)
        wrong = torch.zeros(size, dtype=torch.long, device=device)
        active = torch.ones(size, dtype=torch.bool, device=device)
        lengths = torch.full((size,), length, dtype=torch.long, device=device)
        sequences: list[list[str]] = [[] for _ in range(size)]

        for _ in range(N_LETTERS):
            if not bool(active.any()):
                break
            rows = active.nonzero(as_tuple=True)[0]
            board = torch.where(revealed[rows], letter_ids[rows], torch.full_like(letter_ids[rows], BLANK_ID))
            hidden_mask = ~revealed[rows]
            lives_left = (cfg.max_wrong - wrong[rows]).clamp(min=0)

            score_sum = None
            for model in models:
                with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type == "cuda"):
                    position_logits, set_logits, value_logits = model(board, guessed[rows], lengths[rows])
                scores = score_letters(position_logits.float(), set_logits.float(), value_logits.float(),
                                        hidden_mask, guessed[rows], lives_left, cfg)
                score_sum = scores if score_sum is None else score_sum + scores
            avg_scores = score_sum / len(models)
            picks = avg_scores.argmax(dim=-1)

            guessed[rows, picks] = 1.0
            matches = (letter_ids[rows] == picks.unsqueeze(1)) & is_letter[rows] & ~revealed[rows]
            hit = matches.any(dim=1)
            new_revealed = revealed[rows] | matches
            revealed[rows] = new_revealed
            wrong[rows] = wrong[rows] + (~hit).long()

            picks_cpu = picks.cpu().tolist()
            for local, global_row in enumerate(rows.tolist()):
                sequences[global_row].append(ALPHABET[picks_cpu[local]])

            finished_win = new_revealed.all(dim=1)
            finished_loss = wrong[rows] >= cfg.max_wrong
            active[rows] = ~(finished_win | finished_loss)

        for local, global_index in enumerate(batch_indices):
            guess_strings[global_index] = "".join(sequences[local])
        solved[batch_indices] = revealed.all(dim=1).cpu().numpy()
        wrong_counts[batch_indices] = wrong.cpu().numpy()

        if verbose and (cursor // max(1, cfg.eval_batch_size)) % 25 == 0:
            done = cursor / len(order)
            print(f"  simulated {cursor:>7,}/{len(order):,} ({done:5.1%})  {time.time() - start:6.1f}s")

    return PlayResult(guess_strings, solved, wrong_counts, [])


def evaluate_ensemble(models: Sequence[HangmanNet], words: Sequence[str], cfg: Config, label: str = "eval") -> dict:
    result = play_games_ensemble(models, words, cfg)
    win_rate = float(result.solved.mean())
    report = {"label": label, "words": len(words), "win_rate": win_rate,
              "avg_wrong": float(result.wrong_counts.mean()), "by_length": {}}
    lengths = np.array([len(w) for w in words])
    for bucket in [(1, 4), (5, 6), (7, 8), (9, 10), (11, 13), (14, 99)]:
        mask = (lengths >= bucket[0]) & (lengths <= bucket[1])
        if mask.sum():
            report["by_length"][f"{bucket[0]}-{bucket[1]}"] = (float(result.solved[mask].mean()), int(mask.sum()))
    return report


# --------------------------------------------------------------------------
# Submission
# --------------------------------------------------------------------------


def build_submission(model_or_models, test_words: Sequence[str], cfg: Config, path: str) -> "object":
    import pandas as pd

    if isinstance(model_or_models, (list, tuple)):
        result = play_games_ensemble(model_or_models, test_words, cfg, verbose=True)
    else:
        result = play_games(model_or_models, test_words, cfg, verbose=True)

    frame = pd.DataFrame({
        "word_id": np.arange(len(test_words), dtype=np.int64),
        "guessed_letters_string": result.guess_strings,
    })
    frame.to_csv(path, index=False)
    print(f"\nwrote {path}: {len(frame):,} rows")
    print(f"simulated win rate on this list: {result.solved.mean():.4f}  "
          f"total wrong guesses: {int(result.wrong_counts.sum()):,}")
    return frame


def verify_submission(frame, n_expected: int) -> None:
    """Fail loudly before submitting rather than silently after."""
    assert list(frame.columns) == ["word_id", "guessed_letters_string"], "column names/order wrong"
    assert len(frame) == n_expected, f"expected {n_expected} rows, found {len(frame)}"
    assert frame["word_id"].tolist() == list(range(n_expected)), "word_id must be 0..N-1 in order"

    allowed = set(ALPHABET)
    for guesses in frame["guessed_letters_string"]:
        assert isinstance(guesses, str) and guesses, "empty guess string"
        assert set(guesses) <= allowed, f"illegal character in {guesses!r}"
        assert len(set(guesses)) == len(guesses), f"duplicate guess in {guesses!r}"
    print("submission checks passed: schema, ordering, alphabet, no duplicate guesses")


In [ ]:
import importlib
import hangman
importlib.reload(hangman)

from hangman import (
    Config, set_seed, load_words, split_holdout,
    train, evaluate, evaluate_ensemble, print_report,
    build_submission, verify_submission,
)

## 2. Configuration

In [ ]:
DATA_DIR = "/kaggle/input/competitions/brand-buzzword-hackathon"

cfg = Config(
    train_path=f"{DATA_DIR}/train.txt",
    test_path=f"{DATA_DIR}/test.txt",

    # model capacity
    d_model=320,
    n_lstm_layers=2,
    n_transformer_layers=3,
    n_heads=8,
    dropout=0.15,

    # phase 1
    samples_per_word=8,
    epochs=40,
    batch_size=512,
    lr=3e-4,
    curriculum=True,

    # phase 2
    onpolicy_rounds=5,
    onpolicy_words=100_000,
    onpolicy_epochs=2,
    onpolicy_lr=1e-4,

    # inference
    set_head_weight=0.5,
    use_value_lookahead=True,
    value_weight=0.35,
    eval_batch_size=2048,

    seed=1337,
    num_workers=2,
)

set_seed(cfg.seed)
print(f"device: {cfg.device}")

## 3. Vocabulary

In [ ]:
from collections import Counter

all_words = load_words(cfg.train_path)
train_words, holdout_words = split_holdout(all_words, cfg.holdout_frac, cfg.seed)

print(f"vocabulary       : {len(all_words):,}")
print(f"training words   : {len(train_words):,}")
print(f"held-out words   : {len(holdout_words):,}")

lengths = Counter(len(w) for w in all_words)
for length in sorted(lengths):
    bar = "#" * int(40 * lengths[length] / max(lengths.values()))
    print(f"  {length:>2}: {lengths[length]:>6,} {bar}")

## 4. Training (phase 1 + phase 2)

In [ ]:
import time

started = time.time()
model = train(cfg, train_words, holdout_words)
print(f"\ntotal training time: {(time.time() - started) / 60:.1f} min")

In [ ]:
import torch

torch.save({"state_dict": model.state_dict(), "config": cfg.__dict__}, "hangman_model_seed1337.pt")
print("checkpoint saved")

## 5. Optional: train a second seed for ensembling

Cheap accuracy boost once a single model has plateaued. Skip this cell if you're
short on time -- the submission cell below works with a single model too.

In [ ]:
TRAIN_SECOND_SEED = False  # flip to True if you have GPU time to spare

if TRAIN_SECOND_SEED:
    cfg2 = Config(**{**cfg.__dict__, "seed": 2027})
    model2 = train(cfg2, train_words, holdout_words)
    torch.save({"state_dict": model2.state_dict(), "config": cfg2.__dict__}, "hangman_model_seed2027.pt")
    models_for_submission = [model, model2]
    print_report(evaluate_ensemble(models_for_submission, holdout_words, cfg, "holdout/ensemble"))
else:
    models_for_submission = model

## 6. Final evaluation on unseen words

In [ ]:
print_report(evaluate(model, holdout_words, cfg, "final holdout (never trained on)"))

## 7. Tune `set_head_weight` / `value_weight` on holdout (optional, free)

Pure inference-time sweep -- no retraining. Only uses the holdout split from
`train.txt`, never `test.txt`.

In [ ]:
RUN_SWEEP = False  # flip to True to search; leave off to keep the defaults above

if RUN_SWEEP:
    best = (cfg.set_head_weight, cfg.value_weight, -1.0)
    for shw in [0.3, 0.4, 0.5, 0.6, 0.7]:
        for vw in [0.0, 0.2, 0.35, 0.5]:
            cfg.set_head_weight, cfg.value_weight = shw, vw
            r = evaluate(model, holdout_words, cfg, f"shw={shw} vw={vw}")
            print(f"  set_head_weight={shw}  value_weight={vw}  win_rate={r['win_rate']:.4f}")
            if r["win_rate"] > best[2]:
                best = (shw, vw, r["win_rate"])
    cfg.set_head_weight, cfg.value_weight = best[0], best[1]
    print(f"\nbest: set_head_weight={best[0]}  value_weight={best[1]}  win_rate={best[2]:.4f}")

## 8. Submission

In [ ]:
test_words = load_words(cfg.test_path)
print(f"test words: {len(test_words):,}")

submission = build_submission(models_for_submission, test_words, cfg, "submission.csv")
verify_submission(submission, len(test_words))
submission.head(10)

## 9. Error analysis on held-out words

In [ ]:
from hangman import play_games

result = play_games(model, holdout_words, cfg)
lost = [(w, g) for w, g, s in zip(holdout_words, result.guess_strings, result.solved) if not s]

print(f"lost {len(lost):,} of {len(holdout_words):,} held-out words\n")
for word, guesses in lost[:25]:
    revealed = "".join(c if c in guesses else "_" for c in word)
    missed = "".join(g for g in guesses if g not in word)
    print(f"  {word:<20} board={revealed:<20} guessed={guesses:<16} misses={missed}")